In [52]:
!pip install tsfresh

#Solving a problem with compatibility between versions in my Python envionment. Dask is a dependency of tsfresh, but it seems to be causing issues. Uninstalling it to see if that resolves the problem.
import sys
!{sys.executable} -m pip uninstall -y dask

#Setting up working director
from pathlib import Path

candidate_project_dirs = [
    Path.cwd().parent,                          # if cwd is .../Projects/HoldingPenTime/GP_Model
    Path.cwd() / 'Projects/HoldingPenTime',     # if cwd is repo root
    Path.cwd(),                                 # if cwd is already .../Projects/HoldingPenTime
]

project_dir = next((p.resolve() for p in candidate_project_dirs if (p / 'Data').exists()), None)
if project_dir is None:
    raise FileNotFoundError(f'Could not resolve project_dir from cwd={Path.cwd()}')

data_dir = project_dir / 'Data'
processed_dir = data_dir / 'processed'
problematic_dir = processed_dir / 'Problematic_targetCows'

print("project_dir   :", project_dir)
print("data_dir      :", data_dir)
print("processed_dir :", processed_dir)
print("problematic_dir:", problematic_dir)



[notice] A new release of pip is available: 23.3 -> 26.0.1
[notice] To update, run: C:\Users\tokm0001\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


project_dir   : C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime
data_dir      : C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data
processed_dir : C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data\processed
problematic_dir: C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data\processed\Problematic_targetCows


In [53]:
# This notebook is used to generate the data file "work/Data/processed/Cow_Prob_dataset_L1.csv" 
# for Exp_1 and Exp_2 based on Lactation period 1. The actual analyses are in 

# work/HoldingPenTime/GP_Model/Cow_Detection_L1_Exp_1 .ipynb

# work/HoldingPenTime/GP_Model/Early_Cow_Detection_TS_length_early_150_days_Exp_2.ipynb
# work/HoldingPenTime/GP_Model/Early_Cow_Detection_TS_length_last_150_days_Exp_2.ipynb



In [ ]:
'''
#Gigacow-tools# - data collection for fast/slow learner.
This scripts used for single cow data collection work.
Data Tables: gigacow_filter.csv, lactation_filter.csv, traffic_raw_filter.csv
'''

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
pd.options.mode.chained_assignment = None
from tsfresh import extract_features
from tsfresh import select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_selection.relevance import calculate_relevance_table
from tsfresh.feature_extraction import ComprehensiveFCParameters, MinimalFCParameters
from sklearn.preprocessing import StandardScaler, OneHotEncoder

dataDir = processed_dir
if dataDir is None:
    raise FileNotFoundError(f'Could not find processed data directory. Tried: {candidate_dirs}')

print('Using dataDir:', dataDir)

gigacow_cols = ['Del_Cow_Id', 'FarmName_Pseudo', 'BreedName', 'BirthDate']
lactation_cols = ['Del_Cow_Id', 'FarmName_Pseudo', 'LactationInfoDate', 'LactationNumber', 'DaysInMilk']
print(traffic.Del_Cow_Id.value_counts().nlargest(10))

gigacow = pd.read_csv(dataDir / 'gigacow_filter.csv', encoding='utf-8', usecols=gigacow_cols)
lactation = pd.read_csv(dataDir / 'lactation_filter.csv', encoding='utf-8', usecols=lactation_cols)
traffic = pd.read_csv(dataDir / 'traffic_raw_filter.csv', encoding='utf-8', index_col=False)

print(traffic['Del_Cow_Id'].value_counts().nlargest(10))


Using dataDir: C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data\processed
Del_Cow_Id
12313    18452
13856    11001
487       9433
11787     8646
13719     5404
3711      5232
6906      5041
892       4506
11823     3860
13864      433
Name: count, dtype: int64
Del_Cow_Id
12313    18452
13856    11001
487       9433
11787     8646
13719     5404
3711      5232
6906      5041
892       4506
11823     3860
13864      433
Name: count, dtype: int64


In [55]:
# Select cows with sufficient data points on single lactation periods
# Try to collect cow's data that contain milking events on lactation periods 1
# fetch all milking traffic events for merging
traffic_milking = traffic.TrafficResult.str.contains('kg', regex=False)
all_list = traffic_milking.index[traffic_milking.values == True].tolist()
milking_total = traffic[traffic.index.isin(all_list)]
milking_total.TrafficEventDateTime = pd.to_datetime(milking_total.TrafficEventDateTime)
milking_total['milking_date'] = milking_total.TrafficEventDateTime.dt.date

# convert data type
milking_total.milking_date = pd.to_datetime(milking_total.milking_date)
lactation.LactationInfoDate = pd.to_datetime(lactation.LactationInfoDate)
# merge all milking events with lactation table for filtering
milking_total = milking_total.merge(
    lactation,
    how='left',
    left_on=['FarmName_Pseudo', 'Del_Cow_Id', 'milking_date'],
    right_on=['FarmName_Pseudo', 'Del_Cow_Id', 'LactationInfoDate']
)

def lac_collect(NumLac, milking_total):
    """Generate cow list for multiple lactation periods.

    Args:
        NumLac: The number of lactaion period
        milking_total: A dataframe contains all the milkings events

    Returns:
        A list contain all the cows events with sufficient data points within the lactation periods.
    """
    cow_list = list()
    for num in range(1, NumLac+1):
        # select records that contains lactation period #num
        milking_select1 = milking_total.loc[milking_total['LactationNumber'] == num]
        milking_select1.drop_duplicates(
            subset=['Del_Cow_Id', 'milking_date', 'LactationNumber', 'DaysInMilk'],
            inplace=True
        )
        # drop the anomaly data points
        milking_select1 = milking_select1.loc[milking_select1.DaysInMilk < 400]
        # select sufficient data points on lactation
        selected1 = milking_select1.Del_Cow_Id.value_counts(ascending=True)

        selected1 = selected1.loc[(selected1.values > 150) & (selected1.values < 365)]
        selected_cow_list = selected1.index.to_list()
        if num == 1:
            cow_list = selected_cow_list
        cow_list = list(set(cow_list) & set(selected_cow_list))
        print(len(cow_list), cow_list)
    return cow_list

cow_list = lac_collect(1, milking_total)

5 [487, 11787, 13719, 12313, 3711]


In [ ]:
def countCowAge(birthDate, milkingDate):
    '''
    func: Calculate cows age based on birthDate and milkingDate
    args: 
        birthDate: cow's birth datetime
        milkingDate: milking events datetime
    return: cow age in human years(float)
    '''
    birthDate = pd.to_datetime(birthDate)
    milkingDate = pd.to_datetime(milkingDate)
    days = np.float32(np.datetime64(milkingDate, 'D') - np.datetime64(birthDate, 'D'))
    age = np.around(days/365, 2)
    return age

# select single cow from the traffic table

def data_collector(traffic, gigacow, lactation, cow_id, lacNumList):
    '''
    func: collect features from for a single cow
    args: 
        traffic: traffic data table
        gigacow: gigacow data table
        lactation: lactation data table
        cow_id: gigacow_id of the cow
        lacNumList: a list lactation period number
    return: A dataframe contains all features for a single cow on specfic lactation period
    '''

    traffic_single_cow = traffic.loc[traffic['Del_Cow_Id'] == cow_id]

    traffic_single_cow.sort_values(by='TrafficEventDateTime', inplace=True)
    traffic_single_cow.index = range(len(traffic_single_cow))

    '''
        Extract Milking Event and its most recent traffic event to calculate T2-T1
        T1: Entry time into the Mjolkfalla
        T2: Entry time into the milking robot
        T2-T1: calculate time difference between T2&T1 (i.e., Time spend in Mjolkfalla/holding area)
    '''
    # locate mikling event by searching 'kg' keyword in traffic result
    # the most recent traffic event to milking event should be pre_milking event
    # need to filter out records with gate failure
    track_milking = traffic_single_cow.TrafficResult.str.contains('kg', regex=False)
    milking_index_list = track_milking.index[track_milking.values == True].tolist()
    pre_milking_index_list = [x-1 for x in milking_index_list]
    milking_traffic = traffic_single_cow[traffic_single_cow.index.isin(milking_index_list)]
    pre_milking_traffic = traffic_single_cow[traffic_single_cow.index.isin(pre_milking_index_list)]

    # drop rows that the gate failed to detect cows but have milking result
    # previous area in milking_traffic table should only be Mjolkfalla
    # previous area in pre_milking_traffic table should not be Mjolkfalla
    failed_list_1_milk = milking_traffic.index[milking_traffic['PreviousArea'] == 'Koridor till Sorteringsgrind 2'].tolist()
    failed_list_1_pre = [x-1 for x in failed_list_1_milk]
    failed_list_2_pre = pre_milking_traffic.index[pre_milking_traffic['PreviousArea'] == 'Mjolkfalla'].tolist()
    failed_list_2_milk = [x+1 for x in failed_list_2_pre]
    # traffic result in pre_milking_traffic table should contain Mjolkfalla
    track_pre_milking = pre_milking_traffic.TrafficResult.str.contains('Mjolkfalla', regex=False)
    failed_list_3_pre = track_pre_milking.index[track_pre_milking.values == False].tolist()
    failed_list_3_milk = [x+1 for x in failed_list_3_pre]

    # remove failed records based on index list
    milking_traffic_failed = failed_list_1_milk + failed_list_2_milk + failed_list_3_milk
    pre_milking_traffic_failed = failed_list_1_pre + failed_list_2_pre + failed_list_3_pre
    milking_traffic.drop(axis=0, index=milking_traffic_failed, inplace=True)
    pre_milking_traffic.drop(axis=0, index=pre_milking_traffic_failed, inplace=True)
    # concatenate two tables to track the traffic directly
    all_milking_traffic = pd.concat([milking_traffic, pre_milking_traffic])
    all_milking_traffic.sort_values(by=['TrafficEventDateTime'], inplace=True)
    #rename table columns for merging
    milking_traffic.rename(columns={"TrafficEventDateTime": "MilkingEventDateTime", "TrafficResult": "MilkProduction", "TimeInArea_totalSeconds": "RoundedSecondsTimeInArea"}, inplace=True)
    pre_milking_traffic.rename(columns={"TrafficEventDateTime": "Pre_MilkingEventDateTime", "TimeInArea_totalSeconds": "RoundedSecondsTimeInArea"}, inplace=True)
    # unify the index of two tables
    milking_traffic.index = range(len(milking_traffic))
    pre_milking_traffic.index = range(len(pre_milking_traffic))
    # inert "pre_traffic_milking" to milking traffic table
    milking_traffic.insert(5, 'Pre_MilkingEventDateTime', pre_milking_traffic['Pre_MilkingEventDateTime'])
    # calculate T2-T1
    milking_traffic.MilkingEventDateTime = pd.to_datetime(milking_traffic.MilkingEventDateTime)
    milking_traffic.Pre_MilkingEventDateTime = pd.to_datetime(milking_traffic.Pre_MilkingEventDateTime)
    milking_traffic['timeDelta_Seconds'] = (milking_traffic['MilkingEventDateTime'] - milking_traffic['Pre_MilkingEventDateTime']).dt.total_seconds()

    # extract traffic result(milk production)
    milking_traffic['MilkProduction'].replace(r"[^0-9.,]+"," ", inplace=True, regex=True)
    milking_traffic['MilkProduction'].replace(r"\s*","", inplace=True, regex=True)
    milking_traffic['MilkProduction'].replace(r"[,]+",".", inplace=True, regex=True)
    milking_traffic['MilkProduction'] = milking_traffic['MilkProduction'].astype('float64')

    # merge all the other features into milking_traffic table
    milking_traffic['MilkingDate'] = milking_traffic.MilkingEventDateTime.dt.date
    milking_traffic.MilkingDate = pd.to_datetime(milking_traffic.MilkingDate)
    lactation.LactationInfoDate = pd.to_datetime(lactation.LactationInfoDate)

    single_cow_merge = milking_traffic.merge(
    lactation,
    how='left',
    left_on=['FarmName_Pseudo', 'Del_Cow_Id', 'MilkingDate'],
    right_on=['FarmName_Pseudo', 'Del_Cow_Id', 'LactationInfoDate']
        )
    single_cow_merge = single_cow_merge.merge(
    gigacow,
    how='left',
    left_on=['FarmName_Pseudo', 'Del_Cow_Id'],
    right_on=['FarmName_Pseudo', 'Del_Cow_Id']
        )

    # drop failed data points based on RoundedSecondsTimeInArea & timeDelta_Seconds
    single_cow_merge.drop(single_cow_merge.loc[abs(single_cow_merge.timeDelta_Seconds - single_cow_merge.RoundedSecondsTimeInArea) > 300].index, inplace=True)
    single_cow_merge['TrafficDeviceName'].replace(r"[A-Za-z]+\s*","vms", inplace=True, regex=True)
    # calculate age of cows
    single_cow_merge['Age'] = single_cow_merge.apply(lambda x: countCowAge(x['BirthDate'], x['MilkingEventDateTime']), axis=1)
    single_cow_merge.drop(['BirthDate'], axis=1, inplace=True)
    single_cow_merge.dropna(inplace=True)

    # integrate multiple milking events for a single DIM
    single_cow_merge = single_cow_merge[single_cow_merge.LactationNumber.isin(lacNumList)]
    single_cow_merge.index = range(1,len(single_cow_merge)+1) 
    single_cow_merge.drop(
        [
            'MilkingEventDateTime',
            'Pre_MilkingEventDateTime',
            'dwh_factCowTraffic_Id',
            'MilkingInterval_totalSeconds',
            'RoundedSecondsTimeInArea',
            'PreviousArea',
            'GroupName',
            'LactationInfoDate',
            'TrafficDeviceName'
        ],
        axis=1,
        inplace=True,
        errors='ignore'
    )

    # uncomment following part to get combined milking events for each DIM
    # comb_cows = single_cow_merge.groupby(by=['FarmName_Pseudo', 'Del_Cow_Id', 'MilkingDate', 'LactationNumber', 'DaysInMilk', 'BreedName', 'Age'], sort=False, as_index=False).sum(['MilkProduction', 'timeDelta_Seconds'])
    # single_cow_merge_size = single_cow_merge.groupby(by=['FarmName_Pseudo', 'Del_Cow_Id', 'MilkingDate', 'LactationNumber', 'DaysInMilk', 'BreedName', 'Age'], sort=False, as_index=False).size()
    # comb_cows = pd.concat([comb_cows, single_cow_merge_size['size']], axis=1, ignore_index=False)
    # comb_cows.rename(columns={"MilkProduction": "Total_MilkProduction", "timeDelta_Seconds": "Total_timeDelta_Seconds", "size": "milking_times"}, inplace=True)
    # comb_cows.index = range(1, len(comb_cows)+1)
    # return comb_cows

    single_cow_merge.rename(columns={"MilkProduction": "Total_MilkProduction", "timeDelta_Seconds": "Total_timeDelta_Seconds", "size": "milking_times"}, inplace=True)
    single_cow_merge.index = range(1, len(single_cow_merge)+1)
    return single_cow_merge


In [57]:
"""
labeling cow with problematic/normal(1/0)
"""
threshold_ratio = 0.05
Path(dataDir/'Problematic_targetCows').mkdir(parents=True, exist_ok=True)
def labeling_problematic(threshold_ratio, cow_total): 
    '''
    func: labeling problematic cows
    args: 
        threshold_ratio: threshold ratio for the abnormal event milking events
        cow_total: A dataframe contains all data points for a single cow
    return: problematic cows dataset with label
    '''
    global learner
    total_events = len(cow_total)
    abnoramal_cows = cow_total.loc[cow_total.Total_timeDelta_Seconds > 7200]
    abnoramal_ratio = len(abnoramal_cows)/total_events
    print(abnoramal_ratio)
    if abnoramal_ratio > threshold_ratio:
        problematic = 1 # problematic cow
    else:
        problematic = 0 # normal cow
    cow_total['problematic'] = problematic
    return cow_total

In [58]:
# filter out cows' record start at the middle of the lactation
filter_list = []
for id in cow_list:
    single_cow = data_collector(traffic, gigacow, lactation, id, [1])
    if single_cow.DaysInMilk.min() < 60:
        filter_list.append(id)

print("filtered: ", len(filter_list), filter_list)

filtered:  4 [487, 11787, 13719, 12313]


In [59]:
#Uses the labeling_problematic function to label problematic cows and print distribution of time into a PDF
from matplotlib.backends.backend_pdf import PdfPages
import warnings
pd.set_option('mode.chained_assignment', None)

""" plot the relations between timeDelta and Lactation/DIM(DaysInMilk)
        @@@ Total_timeDelta @@@
    """ 
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with PdfPages(dataDir/'100cows_timecost_scatters_lac1_with_label_scale30000.pdf') as pdf:
        for id in filter_list:
            print("cow_id:", id)
            single_cow_merge = data_collector(traffic, gigacow, lactation, id, [1])
            single_cow_merge = labeling_problematic(0.05, single_cow_merge)
            prob = single_cow_merge.problematic.unique()[0]
            fig1 = plt.figure()
            # fig2 = plt.figure()
            if prob == 1:
                title = "Problematic_Cow_cow_id_"+ str(id)
            else:
                title = "Normal_Cow_cow_id_"+ str(id)
            fig1 = single_cow_merge.loc[single_cow_merge.LactationNumber == 1].plot(x="DaysInMilk", y="Total_timeDelta_Seconds", kind='scatter', title=title+"_Lac1", xlim=[1, 360], ylim=[0, 30000], s=2, c='b')
            # fig2 = single_cow_merge.loc[single_cow_merge.LactationNumber == 2].plot(x="DaysInMilk", y="Total_timeDelta_Seconds", kind='scatter', title=title+"_Lac2", xlim=[1, 360], ylim=[0, 10000], s=2, c='b')
            pdf.savefig(fig1.get_figure())
            # pdf.savefig(fig2.get_figure())
            plt.close()

cow_id: 487
0.06302521008403361
cow_id: 11787
0.017467248908296942
cow_id: 13719
0.05955678670360111
cow_id: 12313
0.004722550177095631


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [60]:
pd.options.mode.chained_assignment = None

mean_totalTimeCost = 0
Path(dataDir/'Problematic_targetCows').mkdir(parents=True, exist_ok=True)
lactationNum = [1]

# save a list of cow data for abnormal cows detection problem
for i, cow_id in enumerate(filter_list):
    single_cow_merge = data_collector(traffic, gigacow, lactation, cow_id, lactationNum)
    mean_totalTimeCost += single_cow_merge.Total_timeDelta_Seconds.mean()
    single_cow_merge = labeling_problematic(threshold_ratio, single_cow_merge)
    problematic = single_cow_merge.problematic.unique()[0]
    if problematic == 1:
        print("This cow is problematic")
    single_cow_merge["id"] = i+1
    single_cow_merge.dropna(inplace=True)
    fileName = 'Problematic_targetCows/cow_' + str(i) + '.csv'
    single_cow_merge.to_csv(dataDir/fileName)
print("num of cows: ", len(cow_list))
print("Mean of total time cost: ", mean_totalTimeCost/len(cow_list))

0.06302521008403361
This cow is problematic
0.017467248908296942
0.05955678670360111
This cow is problematic
0.004722550177095631
num of cows:  5
Mean of total time cost:  1198.8640155073101


In [61]:
""" Data Preparation L1 """
usecols = ['id', 'FarmName_Pseudo', 'Del_Cow_Id', 'Total_MilkProduction', 'Total_timeDelta_Seconds', 'LactationNumber', 'DaysInMilk', 'BreedName', 'Age', 'MilkingDate', 'problematic']

dataDir = problematic_dir

print("Using dataDir:", dataDir)

cow_total = None
filelist = list(dataDir.glob('cow_*.csv'))

for i, file_path in enumerate(filelist):
    single_cow = pd.read_csv(file_path, encoding='utf-8', usecols=usecols)
    single_cow.sort_values(by=['MilkingDate'], inplace=True)
    if i == 0:
        cow_total = single_cow
    else:
        cow_total = pd.concat([cow_total, single_cow], axis=0, ignore_index=True)

if cow_total is None:
    raise ValueError(f'No cow_*.csv files were loaded from {dataDir}')

cow_total.to_csv(dataDir.parent / "Cow_Prob_dataset_L1.csv", index=False)
cow_total

Using dataDir: C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data\processed\Problematic_targetCows


,FarmName_Pseudo,Del_Cow_Id,Total_MilkProduction,Total_timeDelta_Seconds,MilkingDate,LactationNumber,DaysInMilk,BreedName,Age,problematic,id
0,f454e660,487,6.48,2287.0,2021-12-13,1.0,3.0,02 SLB,2.24,1,1
1,f454e660,487,7.79,14313.0,2021-12-14,1.0,4.0,02 SLB,2.24,1,1
2,f454e660,487,8.19,10286.0,2021-12-14,1.0,4.0,02 SLB,2.24,1,1
3,f454e660,487,7.59,28011.0,2021-12-15,1.0,5.0,02 SLB,2.25,1,1
4,f454e660,487,10.89,17405.0,2021-12-15,1.0,5.0,02 SLB,2.25,1,1
...,...,...,...,...,...,...,...,...,...,...,...
2974,a624fb9a,12313,9.07,215.0,2022-04-09,1.0,280.0,01 SRB,2.62,0,4
2975,a624fb9a,12313,10.42,65.0,2022-04-09,1.0,280.0,01 SRB,2.62,0,4
2976,a624fb9a,12313,10.09,330.0,2022-04-09,1.0,280.0,01 SRB,2.62,0,4
2977,a624fb9a,12313,8.16,442.0,2022-04-10,1.0,281.0,01 SRB,2.62,0,4


In [62]:
print("dataDir:", dataDir)
print("exists:", Path(dataDir).exists())
print("files:", list(Path(dataDir).glob('cow_*.csv'))[:10])

dataDir: C:\projects\HoldingPenTime-v2-clean\Projects\HoldingPenTime\Data\processed\Problematic_targetCows
exists: True
files: [WindowsPath('C:/projects/HoldingPenTime-v2-clean/Projects/HoldingPenTime/Data/processed/Problematic_targetCows/cow_0.csv'), WindowsPath('C:/projects/HoldingPenTime-v2-clean/Projects/HoldingPenTime/Data/processed/Problematic_targetCows/cow_1.csv'), WindowsPath('C:/projects/HoldingPenTime-v2-clean/Projects/HoldingPenTime/Data/processed/Problematic_targetCows/cow_2.csv'), WindowsPath('C:/projects/HoldingPenTime-v2-clean/Projects/HoldingPenTime/Data/processed/Problematic_targetCows/cow_3.csv')]
